<a href="https://colab.research.google.com/github/sinamahdavi/aml-2025-mistake-detection/blob/sanam/notebooks/substep3_task_graph_matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Substep 3: Task-Graph Encoding + Step Matching

## Overview

This notebook implements **Substep 3** of the extension:
1. **Task-Graph Encoding**: Encode task graph step descriptions using EgoVLP textual encoder
2. **Step Matching**: Match visual steps to task graph nodes using Hungarian matching algorithm
3. **Feature Update**: Update matched node features with learnable projection

## Pipeline

- **Step 1**: Load and understand task graph structure
- **Step 2**: Set up EgoVLP text encoder
- **Step 3**: Encode all task graph nodes
- **Step 4**: Prepare visual step embeddings (from Substep 1)
- **Step 5**: Compute similarity matrix
- **Step 6**: Hungarian matching algorithm
- **Step 7**: Create learnable projection layer
- **Step 8**: Update matched node features
- **Step 9**: Handle unmatched nodes/steps
- **Step 10**: Save processed task graphs

---


## Setup

### 1. Install Required Packages


In [1]:
# Install required packages
!pip install -q torch-geometric
!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git

print("✅ Packages installed!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
✅ Packages installed!


### 2. Clone Repository (if needed)


In [2]:
%cd /content
!rm -rf code

!git clone --recursive -b sanam https://github.com/sinamahdavi/aml-2025-mistake-detection.git code
%cd code

/content
Cloning into 'code'...
remote: Enumerating objects: 771, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 771 (delta 20), reused 95 (delta 10), pack-reused 622 (from 1)
Receiving objects: 100% (771/771), 6.91 MiB | 975.00 KiB/s, done.
Resolving deltas: 100% (440/440), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 3.21 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'
/content/code


### 3. Mount Google Drive (Colab only)


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 4. Setup EgoVLP Features Path


In [4]:
# Setup EgoVLP features path
# Shared folder: https://drive.google.com/drive/folders/1Wb7fgwS94VZ27weF4UGQDy9GaTQWHW_C?usp=drive_link
# Add as shortcut to "My Drive" if not already done

import os
from pathlib import Path

# Search for egovlp directory in MyDrive
EGOVLP_FEATURES_PATH = None
if os.path.exists('/content/drive/MyDrive'):
    for item in os.listdir('/content/drive/MyDrive'):
        item_path = os.path.join('/content/drive/MyDrive', item)
        if os.path.isdir(item_path) and 'egovlp' in item.lower():
            # Check if it contains .npz files
            try:
                files = [f for f in os.listdir(item_path) if f.endswith('.npz')]
                if len(files) > 0:
                    EGOVLP_FEATURES_PATH = item_path
                    print(f"✅ Found EgoVLP features: {item_path} ({len(files)} files)")
                    break
            except:
                pass

# Default path if not found
if not EGOVLP_FEATURES_PATH:
    EGOVLP_FEATURES_PATH = "/content/drive/MyDrive/egovlp"
    print(f"⚠️  Using default path: {EGOVLP_FEATURES_PATH}")
    print("   If features not found, add shortcut from: https://drive.google.com/drive/folders/1Wb7fgwS94VZ27weF4UGQDy9GaTQWHW_C?usp=drive_link")

# Set FEATURES_DIR for use in configuration and later cells
FEATURES_DIR = Path(EGOVLP_FEATURES_PATH)
print(f"\n✅ Features path set: {FEATURES_DIR}")


✅ Found EgoVLP features: /content/drive/MyDrive/egovlp (384 files)

✅ Features path set: /content/drive/MyDrive/egovlp


### 5. Imports


In [5]:
# Core imports
import os
import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# Scipy for Hungarian matching
from scipy.optimize import linear_sum_assignment

# CLIP (for text encoding - will be used for EgoVLP text encoder)
try:
    import clip
    CLIP_AVAILABLE = True
    print("✅ CLIP loaded")
except ImportError:
    print("⚠️  CLIP not found, installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/openai/CLIP.git"])
    import clip
    CLIP_AVAILABLE = True
    print("✅ CLIP installed and loaded")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print(f"\n✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


✅ CLIP loaded

✅ All imports successful!
PyTorch version: 2.9.0+cu126
Device: cuda


## Configuration

Set up paths and settings. Uses paths from the existing extension notebook.


In [6]:
# Paths (matching extension_complete notebook structure)
TASK_GRAPHS_DIR = Path("annotations/task_graphs")
ANNOTATIONS_PATH = Path("annotations/annotation_json/complete_step_annotations.json")
SPLIT_FILE = Path("er_annotations/recordings_combined_splits.json")

# Features path will be set in cell 8 (Setup EgoVLP Features Path)
# Use it if already set, otherwise will be set later
try:
    FEATURES_DIR = Path(EGOVLP_FEATURES_PATH)
except NameError:
    FEATURES_DIR = None  # Will be set in cell 8

OUTPUT_DIR = Path("extension_results/substep3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("✅ Configuration loaded")
print(f"   Task graphs directory: {TASK_GRAPHS_DIR}")
if FEATURES_DIR:
    print(f"   Features directory: {FEATURES_DIR}")
else:
    print(f"   Features directory: (will be set in cell 8)")
print(f"   Output directory: {OUTPUT_DIR}")
print(f"   Device: {DEVICE}")

# Check if paths exist
print("\n📂 Checking paths...")
print(f"   Task graphs exist: {TASK_GRAPHS_DIR.exists()}")
print(f"   Annotations exist: {ANNOTATIONS_PATH.exists()}")
print(f"   Split file exists: {SPLIT_FILE.exists()}")
if FEATURES_DIR:
    print(f"   Features exist: {FEATURES_DIR.exists()}")
else:
    print(f"   Features: (check in cell 8)")


✅ Configuration loaded
   Task graphs directory: annotations/task_graphs
   Features directory: /content/drive/MyDrive/egovlp
   Output directory: extension_results/substep3
   Device: cuda

📂 Checking paths...
   Task graphs exist: True
   Annotations exist: True
   Split file exists: True
   Features exist: True


## Step 1: Load and Understand Task Graph Structure

In this step, we:
1. Load task graphs from `annotations/task_graphs/`
2. Inspect the JSON structure (nodes, edges, step descriptions)
3. Map each recipe to its task graph


In [7]:
# Load data splits and extract recipe IDs
with open(SPLIT_FILE, 'r') as f:
    splits = json.load(f)

all_recording_ids = splits['train'] + splits['val'] + splits['test']
recipe_ids = sorted(set(recording_id.split('_')[0] for recording_id in all_recording_ids))

print(f"✅ Loaded {len(all_recording_ids)} recordings ({len(recipe_ids)} unique recipes)")

# Try to extract recipe names from annotations (optional)
recipe_name_to_id = {}
try:
    with open(ANNOTATIONS_PATH, 'r') as f:
        annotations = json.load(f)
    for recording_id, data in annotations.items():
        recipe_id = recording_id.split('_')[0]
        if 'recipe_name' in data:
            recipe_name = data['recipe_name'].lower().replace(' ', '')
            recipe_name_to_id[recipe_name] = recipe_id
        elif 'recipe' in data:
            recipe_name = str(data['recipe']).lower().replace(' ', '')
            recipe_name_to_id[recipe_name] = recipe_id
except:
    pass


✅ Loaded 383 recordings (24 unique recipes)


In [8]:
# Check task graphs directory structure
if not TASK_GRAPHS_DIR.exists():
    print(f"❌ Task graphs directory not found: {TASK_GRAPHS_DIR}")
else:
    task_graph_files = list(TASK_GRAPHS_DIR.glob("*.json"))
    print(f"✅ Found {len(task_graph_files)} task graph files")

    if len(task_graph_files) > 0:
        # Inspect one example to understand structure
        with open(task_graph_files[0], 'r') as f:
            example_graph = json.load(f)

        print(f"📊 Task graph structure: {list(example_graph.keys())}")

        # Show structure of main fields
        for key, value in example_graph.items():
            if isinstance(value, (list, dict)):
                print(f"   - {key}: {type(value).__name__} with {len(value)} items")
            else:
                print(f"   - {key}: {type(value).__name__}")


✅ Found 24 task graph files
📊 Task graph structure: ['steps', 'edges']
   - steps: dict with 19 items
   - edges: list with 24 items


In [9]:
# Inspect a few task graphs to understand structure
if TASK_GRAPHS_DIR.exists() and len(task_graph_files) > 0:
    example_graphs = {}
    for graph_file in task_graph_files[:3]:
        with open(graph_file, 'r') as f:
            graph = json.load(f)
        example_graphs[graph_file.stem] = graph

        # Show structure
        print(f"\n📄 {graph_file.stem}:")
        if 'steps' in graph:
            steps = graph['steps']
            step_type = 'dict' if isinstance(steps, dict) else 'list'
            print(f"   Steps: {step_type} with {len(steps)} items")
        elif 'nodes' in graph:
            nodes = graph['nodes']
            node_type = 'dict' if isinstance(nodes, dict) else 'list'
            print(f"   Nodes: {node_type} with {len(nodes)} items")
        else:
            print(f"   Keys: {list(graph.keys())}")



📄 spicytunaavocadowraps:
   Steps: dict with 19 items

📄 buttercorncup:
   Steps: dict with 14 items

📄 tomatochutney:
   Steps: dict with 21 items


In [10]:
# Try to create mapping from recipe_id to task graph file
# (Task graphs are named by recipe name, not ID, so mapping may not be possible)

# Ensure recipe_name_to_id is available
try:
    _ = recipe_name_to_id
except NameError:
    recipe_name_to_id = {}

recipe_to_graph = {}
task_graph_files = list(TASK_GRAPHS_DIR.glob("*.json")) if TASK_GRAPHS_DIR.exists() else []

# Try to map: check if task graphs contain recipe_id, or match by name
for graph_file in task_graph_files:
    graph_filename = graph_file.stem.lower()

    # Check if task graph contains recipe_id
    try:
        with open(graph_file, 'r') as f:
            graph_data = json.load(f)
        if 'recipe_id' in graph_data:
            recipe_id = str(graph_data['recipe_id'])
            if recipe_id in recipe_ids:
                recipe_to_graph[recipe_id] = graph_file
                continue
    except:
        pass

    # Try matching by recipe name
    if graph_filename in recipe_name_to_id:
        recipe_id = recipe_name_to_id[graph_filename]
        if recipe_id not in recipe_to_graph:
            recipe_to_graph[recipe_id] = graph_file

# Create recording-to-graph mapping
recording_to_graph = {rid: recipe_to_graph[rid.split('_')[0]]
                      for rid in all_recording_ids
                      if rid.split('_')[0] in recipe_to_graph}

print(f"✅ Mapped {len(recipe_to_graph)}/{len(recipe_ids)} recipes, {len(recording_to_graph)}/{len(all_recording_ids)} recordings")
if len(recipe_to_graph) == 0:
    print("   Note: Task graphs will be loaded by filename in later steps.")


✅ Mapped 0/24 recipes, 0/383 recordings
   Note: Task graphs will be loaded by filename in later steps.


### Quick Check: Inspect Step Structure

Before moving to Step 2, let's verify what fields are in the task graph steps to know what to encode.


In [11]:
# Inspect step structure - steps are dictionaries with string values (step descriptions)
# Get task graph names first
task_graph_names = [f.stem for f in TASK_GRAPHS_DIR.glob("*.json")] if TASK_GRAPHS_DIR.exists() else []

if len(task_graph_names) > 0:
    # Helper function to load task graph by name
    def load_task_graph_by_name(graph_filename: str):
        """Load task graph by filename (without .json extension)."""
        graph_path = TASK_GRAPHS_DIR / f"{graph_filename}.json"
        if graph_path.exists():
            with open(graph_path, 'r') as f:
                return json.load(f)
        return None

    test_graph = load_task_graph_by_name(task_graph_names[0])
    if test_graph and 'steps' in test_graph:
        steps = test_graph['steps']
        if isinstance(steps, dict) and len(steps) > 0:
            # Show first few steps
            print(f"📋 Task graph: {task_graph_names[0]}")
            print(f"   Total steps: {len(steps)}")
            print(f"\n   Sample steps:")
            for i, (step_id, step_description) in enumerate(list(steps.items())[:3]):
                print(f"   Step {step_id}: {step_description}")
            print(f"\n✅ Steps are strings (descriptions) - ready to encode with EgoVLP text encoder!")

📋 Task graph: spicytunaavocadowraps
   Total steps: 19

   Sample steps:
   Step 0: START
   Step 1: place-place avocado slices on each leaf
   Step 2: Season-season 1/4 tsp pepper on the bowl

✅ Steps are strings (descriptions) - ready to encode with EgoVLP text encoder!


---

## Substep 1: Recipe Step Localization

Extract step-level embeddings from video features using step boundaries. We'll use ground-truth step boundaries from annotations to extract step embeddings by averaging EgoVLP features within each step's time boundaries.


In [12]:
# Substep 1: Extract step embeddings from ground-truth step boundaries
print("=" * 70)
print("📹 Substep 1: Recipe Step Localization")
print("=" * 70)

# Load ground-truth step annotations
step_annotations_path = Path("annotations/annotation_json/step_annotations.json")
if not step_annotations_path.exists():
    print(f"❌ Step annotations not found: {step_annotations_path}")
    print("   Make sure annotations submodule is cloned and up to date")
    visual_step_embeddings = {}
else:
    print(f"📥 Loading step annotations from: {step_annotations_path}")
    with open(step_annotations_path, 'r') as f:
        step_annotations = json.load(f)

    print(f"✅ Loaded step annotations for {len(step_annotations)} recordings")

    # Extract step embeddings
    print(f"\n🔧 Extracting step embeddings from EgoVLP features...")
    print(f"   Features directory: {FEATURES_DIR}")

    if not FEATURES_DIR or not FEATURES_DIR.exists():
        print(f"❌ EgoVLP features directory not found: {FEATURES_DIR}")
        print("   Please set up EgoVLP features path in the setup cells above")
        visual_step_embeddings = {}
    else:
        visual_step_embeddings = {}
        recording_info = {}

        # Get all recording IDs from splits
        all_recording_ids = splits['train'] + splits['val'] + splits['test']

        print(f"   Processing {len(all_recording_ids)} recordings...")

        for recording_id in tqdm(all_recording_ids, desc="Extracting embeddings"):
            # Load EgoVLP features for this recording
            feature_file = FEATURES_DIR / f"{recording_id}_360p.mp4_1s_1s.npz"

            if not feature_file.exists():
                continue

            try:
                # Load features
                feature_data = np.load(feature_file)
                # EgoVLP .npz files use 'arr_0' as the key (not 'features')
                if 'arr_0' in feature_data:
                    features = feature_data['arr_0']  # Shape: (T, 768) where T is number of 1-second clips
                elif 'features' in feature_data:
                    features = feature_data['features']
                else:
                    # Try to get the first array in the file
                    keys = list(feature_data.keys())
                    if len(keys) > 0:
                        features = feature_data[keys[0]]
                    else:
                        raise KeyError(f"No data found in {feature_file}")

                # Get step boundaries for this recording
                if recording_id not in step_annotations:
                    continue

                # Structure: step_annotations[recording_id] is a dict with 'steps' key
                recording_data = step_annotations[recording_id]
                if isinstance(recording_data, dict) and 'steps' in recording_data:
                    recording_steps = recording_data['steps']
                elif isinstance(recording_data, list):
                    # If it's directly a list of steps
                    recording_steps = recording_data
                else:
                    # Skip if structure is unexpected
                    continue

                # Extract step embeddings by averaging features within each step boundary
                step_embeddings = []
                step_boundaries = []

                for step in recording_steps:
                    start_time = step.get('start_time', 0)
                    end_time = step.get('end_time', 0)

                    # Convert time to frame indices (features are at 1 fps)
                    start_frame = int(start_time)
                    end_frame = int(end_time) + 1  # +1 to include the end frame

                    # Ensure indices are within bounds
                    start_frame = max(0, min(start_frame, len(features) - 1))
                    end_frame = max(start_frame + 1, min(end_frame, len(features)))

                    if end_frame > start_frame:
                        # Average features within step boundaries
                        step_features = features[start_frame:end_frame]  # (N_frames, 768)
                        step_embedding = np.mean(step_features, axis=0)  # (768,)
                        step_embeddings.append(step_embedding)
                        step_boundaries.append((start_time, end_time))

                if len(step_embeddings) > 0:
                    visual_step_embeddings[recording_id] = np.array(step_embeddings)  # (N_steps, 768)
                    recording_info[recording_id] = {
                        'num_steps': len(step_embeddings),
                        'step_boundaries': step_boundaries
                    }

            except Exception as e:
                print(f"   ⚠️  Error processing {recording_id}: {e}")
                continue

        # Save embeddings
        if len(visual_step_embeddings) > 0:
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            embeddings_path = OUTPUT_DIR / "step1_groundtruth_embeddings.npz"

            print(f"\n💾 Saving embeddings to: {embeddings_path}")
            np.savez_compressed(embeddings_path, **visual_step_embeddings)

            # Save metadata
            metadata_path = OUTPUT_DIR / "step1_groundtruth_metadata.json"
            with open(metadata_path, 'w') as f:
                json.dump(recording_info, f, indent=2)

            print(f"✅ Extracted embeddings for {len(visual_step_embeddings)} recordings")
            print(f"   Average steps per recording: {np.mean([len(e) for e in visual_step_embeddings.values()]):.1f}")
            print(f"   Embeddings saved to: {embeddings_path}")

            # Show sample statistics
            sample_recording = list(visual_step_embeddings.keys())[0]
            sample_embeddings = visual_step_embeddings[sample_recording]
            print(f"\n📊 Sample statistics:")
            print(f"   Recording: {sample_recording}")
            print(f"   Number of steps: {len(sample_embeddings)}")
            print(f"   Embedding shape: {sample_embeddings.shape}")
            print(f"   Embedding dtype: {sample_embeddings.dtype}")
        else:
            print(f"\n⚠️  No embeddings extracted. Check:")
            print(f"   1. EgoVLP features are available in {FEATURES_DIR}")
            print(f"   2. Step annotations are available")
            print(f"   3. Recording IDs match between features and annotations")


📹 Substep 1: Recipe Step Localization
📥 Loading step annotations from: annotations/annotation_json/step_annotations.json
✅ Loaded step annotations for 384 recordings

🔧 Extracting step embeddings from EgoVLP features...
   Features directory: /content/drive/MyDrive/egovlp
   Processing 383 recordings...


Extracting embeddings: 100%|██████████| 383/383 [00:43<00:00,  8.89it/s]



💾 Saving embeddings to: extension_results/substep3/step1_groundtruth_embeddings.npz
✅ Extracted embeddings for 383 recordings
   Average steps per recording: 14.9
   Embeddings saved to: extension_results/substep3/step1_groundtruth_embeddings.npz

📊 Sample statistics:
   Recording: 1_19
   Number of steps: 12
   Embedding shape: (12, 768)
   Embedding dtype: float32


---

## Step 2: Set up EgoVLP Text Encoder

EgoVLP uses CLIP architecture, so we'll use CLIP's text encoder to encode task graph step descriptions.



In [13]:
# Load CLIP model (EgoVLP uses CLIP architecture)
import clip

print("=" * 70)
print("🔤 Loading CLIP Text Encoder (EgoVLP uses CLIP architecture)")
print("=" * 70)

# Load CLIP model - using ViT-B/32 as it's commonly used with EgoVLP
model_name = "ViT-B/32"
print(f"📥 Loading CLIP model: {model_name}...")
clip_model, clip_preprocess = clip.load(model_name, device=DEVICE)
clip_model.eval()

print(f"✅ CLIP model loaded on {DEVICE}")
print(f"   Text encoder ready for encoding task graph step descriptions")

# Test encoding a sample text
test_text = "place-place avocado slices on each leaf"
with torch.no_grad():
    text_tokens = clip.tokenize([test_text]).to(DEVICE)
    test_embedding = clip_model.encode_text(text_tokens)
    print(f"\n🧪 Test encoding:")
    print(f"   Text: '{test_text}'")
    print(f"   Embedding shape: {test_embedding.shape}")
    print(f"   Embedding dim: {test_embedding.shape[1]}")


🔤 Loading CLIP Text Encoder (EgoVLP uses CLIP architecture)
📥 Loading CLIP model: ViT-B/32...


100%|███████████████████████████████████████| 338M/338M [00:13<00:00, 27.2MiB/s]


✅ CLIP model loaded on cuda
   Text encoder ready for encoding task graph step descriptions

🧪 Test encoding:
   Text: 'place-place avocado slices on each leaf'
   Embedding shape: torch.Size([1, 512])
   Embedding dim: 512


### Encode All Task Graph Step Descriptions

Now we'll encode all step descriptions from all task graphs and store them for matching.


In [14]:
# Encode all task graph step descriptions
print("=" * 70)
print("📝 Encoding All Task Graph Step Descriptions")
print("=" * 70)

# Helper function to get all task graph names
def get_all_task_graph_names():
    """Get list of all task graph filenames."""
    return [f.stem for f in TASK_GRAPHS_DIR.glob("*.json")] if TASK_GRAPHS_DIR.exists() else []

# Helper function to load task graph by name
def load_task_graph_by_name(graph_filename: str):
    """Load task graph by filename (without .json extension)."""
    graph_path = TASK_GRAPHS_DIR / f"{graph_filename}.json"
    if graph_path.exists():
        with open(graph_path, 'r') as f:
            return json.load(f)
    return None

# Dictionary to store encoded step descriptions
# Structure: {graph_name: {step_id: embedding_tensor}}
task_graph_embeddings = {}

# Collect all step descriptions from all task graphs
all_step_descriptions = []  # List of (graph_name, step_id, description)
task_graph_names = get_all_task_graph_names()

print(f"📂 Processing {len(task_graph_names)} task graphs...")

for graph_name in task_graph_names:
    graph = load_task_graph_by_name(graph_name)
    if graph and 'steps' in graph:
        steps = graph['steps']
        if isinstance(steps, dict):
            for step_id, description in steps.items():
                if isinstance(description, str) and description.strip():
                    all_step_descriptions.append((graph_name, step_id, description.strip()))

print(f"✅ Collected {len(all_step_descriptions)} step descriptions")

# Encode in batches for efficiency
BATCH_SIZE = 32
print(f"\n🔄 Encoding {len(all_step_descriptions)} descriptions in batches of {BATCH_SIZE}...")

for i in range(0, len(all_step_descriptions), BATCH_SIZE):
    batch = all_step_descriptions[i:i+BATCH_SIZE]
    descriptions = [desc for _, _, desc in batch]

    # Tokenize and encode
    with torch.no_grad():
        text_tokens = clip.tokenize(descriptions).to(DEVICE)
        embeddings = clip_model.encode_text(text_tokens)
        embeddings = embeddings.cpu()  # Move to CPU to save GPU memory

    # Store embeddings
    for j, (graph_name, step_id, _) in enumerate(batch):
        if graph_name not in task_graph_embeddings:
            task_graph_embeddings[graph_name] = {}
        task_graph_embeddings[graph_name][step_id] = embeddings[j]

    if (i // BATCH_SIZE + 1) % 10 == 0:
        print(f"   Processed {i + len(batch)}/{len(all_step_descriptions)} descriptions...")

print(f"\n✅ Encoding complete!")
print(f"   Encoded {len(task_graph_embeddings)} task graphs")
print(f"   Total step embeddings: {sum(len(steps) for steps in task_graph_embeddings.values())}")

# Show sample statistics
if task_graph_embeddings:
    sample_graph = list(task_graph_embeddings.keys())[0]
    sample_steps = task_graph_embeddings[sample_graph]
    sample_embedding = list(sample_steps.values())[0]
    print(f"\n📊 Sample statistics:")
    print(f"   Task graph: {sample_graph}")
    print(f"   Number of steps: {len(sample_steps)}")
    print(f"   Embedding shape: {sample_embedding.shape}")
    print(f"   Embedding dtype: {sample_embedding.dtype}")


📝 Encoding All Task Graph Step Descriptions
📂 Processing 24 task graphs...
✅ Collected 406 step descriptions

🔄 Encoding 406 descriptions in batches of 32...
   Processed 320/406 descriptions...

✅ Encoding complete!
   Encoded 24 task graphs
   Total step embeddings: 406

📊 Sample statistics:
   Task graph: spicytunaavocadowraps
   Number of steps: 19
   Embedding shape: torch.Size([512])
   Embedding dtype: torch.float16


In [29]:
# Save encoded task graph embeddings to disk
print("=" * 70)
print("💾 Saving Encoded Task Graph Embeddings")
print("=" * 70)

if task_graph_embeddings and len(task_graph_embeddings) > 0:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    embeddings_path = OUTPUT_DIR / "encoded_task_graph_embeddings.npz"

    # Convert tensors to numpy arrays for saving
    embeddings_dict = {}
    for graph_name, step_embeddings in task_graph_embeddings.items():
        # Convert each step embedding to numpy
        step_dict = {}
        for step_id, embedding_tensor in step_embeddings.items():
            # Convert tensor to numpy if needed
            if isinstance(embedding_tensor, torch.Tensor):
                step_dict[step_id] = embedding_tensor.cpu().numpy()
            else:
                step_dict[step_id] = embedding_tensor

        # Store as nested dictionary (will be saved as structured array)
        embeddings_dict[graph_name] = step_dict

    # Save embeddings
    # Note: npz format can handle nested dictionaries, but we'll flatten for easier loading
    # Save as: {graph_name_step_id: embedding}
    flattened_embeddings = {}
    for graph_name, step_embeddings in embeddings_dict.items():
        for step_id, embedding in step_embeddings.items():
            key = f"{graph_name}_{step_id}"
            flattened_embeddings[key] = embedding

    np.savez_compressed(embeddings_path, **flattened_embeddings)

    # Also save metadata (graph structure)
    metadata_path = OUTPUT_DIR / "encoded_task_graph_metadata.json"
    metadata = {}
    for graph_name, step_embeddings in task_graph_embeddings.items():
        metadata[graph_name] = {
            'num_steps': len(step_embeddings),
            'step_ids': list(step_embeddings.keys()),
            'embedding_dim': step_embeddings[list(step_embeddings.keys())[0]].shape[0] if step_embeddings else 0
        }

    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"✅ Saved encoded task graph embeddings:")
    print(f"   Embeddings: {embeddings_path}")
    print(f"   Metadata: {metadata_path}")
    print(f"   Total graphs: {len(task_graph_embeddings)}")
    print(f"   Total embeddings: {sum(len(steps) for steps in task_graph_embeddings.values())}")
else:
    print("⚠️  No task graph embeddings to save")

💾 Saving Encoded Task Graph Embeddings
✅ Saved encoded task graph embeddings:
   Embeddings: extension_results/substep3/encoded_task_graph_embeddings.npz
   Metadata: extension_results/substep3/encoded_task_graph_metadata.json
   Total graphs: 24
   Total embeddings: 406


---

## Step 3: Prepare Visual Step Embeddings (from Substep 1)

Load the visual step embeddings that were extracted in Substep 1 using ActionFormer or ground-truth boundaries.


In [15]:
# Load visual step embeddings from Substep 1
print("=" * 70)
print("📹 Loading Visual Step Embeddings (from Substep 1)")
print("=" * 70)

# Search for embeddings in multiple locations
possible_paths = [
    # Local paths
    Path("extension_results/step1_actionformer_embeddings.npz"),
    Path("extension_results/step1_groundtruth_embeddings.npz"),
    # Google Drive paths (common locations)
    Path("/content/drive/MyDrive/extension_results/step1_actionformer_embeddings.npz"),
    Path("/content/drive/MyDrive/extension_results/step1_groundtruth_embeddings.npz"),
    Path("/content/drive/MyDrive/AML/extension_results/step1_actionformer_embeddings.npz"),
    Path("/content/drive/MyDrive/AML/extension_results/step1_groundtruth_embeddings.npz"),
]

# Also search recursively in common directories
search_dirs = [
    Path("extension_results"),
    Path("/content/drive/MyDrive"),
    Path("/content/drive/MyDrive/AML"),
]

visual_step_embeddings = {}
embedding_type = None
found_path = None

# First, try the explicit paths
for path in possible_paths:
    if path.exists():
        found_path = path
        break

# If not found, search recursively
if not found_path:
    print("🔍 Searching for embeddings in common locations...")
    for search_dir in search_dirs:
        if search_dir.exists():
            for pattern in ["*actionformer*embeddings*.npz", "*groundtruth*embeddings*.npz", "*step1*embeddings*.npz"]:
                matches = list(search_dir.rglob(pattern))
                if matches:
                    found_path = matches[0]
                    print(f"   Found: {found_path}")
                    break
            if found_path:
                break

# Load embeddings if found
if found_path:
    print(f"📥 Loading step embeddings from: {found_path}")
    try:
        data = np.load(found_path, allow_pickle=True)
        visual_step_embeddings = dict(data)

        # Determine embedding type from filename
        if "actionformer" in str(found_path).lower():
            embedding_type = "ActionFormer"
        elif "groundtruth" in str(found_path).lower() or "ground-truth" in str(found_path).lower():
            embedding_type = "Ground-truth"
        else:
            embedding_type = "Unknown"

        print(f"✅ Loaded {len(visual_step_embeddings)} recordings with {embedding_type} embeddings")
    except Exception as e:
        print(f"❌ Error loading embeddings: {e}")
        visual_step_embeddings = {}
else:
    print("⚠️  Step embeddings not found!")
    print(f"\n   Searched in:")
    for path in possible_paths[:2]:
        print(f"   - {path}")
    print(f"\n   Also searched recursively in:")
    for search_dir in search_dirs:
        if search_dir.exists():
            print(f"   - {search_dir}")

    print(f"\n💡 Options:")
    print(f"   1. Run Substep 1 in extension_complete notebook to generate embeddings")
    print(f"   2. If embeddings are in a different location, update the path below:")
    print(f"      CUSTOM_EMBEDDINGS_PATH = '/path/to/your/embeddings.npz'")
    print(f"   3. If embeddings are on Google Drive, make sure Drive is mounted")

    # Allow custom path (user can uncomment and set)
    # CUSTOM_EMBEDDINGS_PATH = "/content/drive/MyDrive/path/to/embeddings.npz"
    # if Path(CUSTOM_EMBEDDINGS_PATH).exists():
    #     found_path = Path(CUSTOM_EMBEDDINGS_PATH)
    #     # ... load code ...

# Show statistics
if visual_step_embeddings:
    total_steps = sum(len(emb) for emb in visual_step_embeddings.values())
    avg_steps = total_steps / len(visual_step_embeddings) if len(visual_step_embeddings) > 0 else 0

    # Check embedding dimensions
    sample_recording = list(visual_step_embeddings.keys())[0]
    sample_embeddings = visual_step_embeddings[sample_recording]

    print(f"\n📊 Statistics:")
    print(f"   Embedding type: {embedding_type}")
    print(f"   Total recordings: {len(visual_step_embeddings)}")
    print(f"   Total visual steps: {total_steps}")
    print(f"   Average steps per recording: {avg_steps:.1f}")
    print(f"   Embedding shape per step: {sample_embeddings.shape[1:] if len(sample_embeddings.shape) > 1 else sample_embeddings.shape}")
    print(f"   Embedding dtype: {sample_embeddings.dtype}")

    # Show sample
    print(f"\n📋 Sample recording: {sample_recording}")
    print(f"   Number of visual steps: {len(sample_embeddings)}")
    print(f"   Step embeddings shape: {sample_embeddings.shape}")


📹 Loading Visual Step Embeddings (from Substep 1)
🔍 Searching for embeddings in common locations...
   Found: extension_results/substep3/step1_groundtruth_embeddings.npz
📥 Loading step embeddings from: extension_results/substep3/step1_groundtruth_embeddings.npz
✅ Loaded 383 recordings with Ground-truth embeddings

📊 Statistics:
   Embedding type: Ground-truth
   Total recordings: 383
   Total visual steps: 5691
   Average steps per recording: 14.9
   Embedding shape per step: (768,)
   Embedding dtype: float32

📋 Sample recording: 1_19
   Number of visual steps: 12
   Step embeddings shape: (12, 768)


---

## Step 4: Compute Similarity Matrix

For each recording, we'll compute a similarity matrix between:
- **Visual steps** (from Substep 1): shape `(N_visual_steps, 768)` - EgoVLP features
- **Task graph nodes** (from Step 2): shape `(N_graph_nodes, 512)` - CLIP text embeddings

**Note:** Since dimensions differ (768 vs 512), we'll need to:
1. Project visual embeddings to 512-dim, OR
2. Project text embeddings to 768-dim, OR  
3. Use a learned projection layer

For now, we'll project visual embeddings to match text embedding dimension using a simple linear layer.


In [16]:
# Check if we have the required data
if 'visual_step_embeddings' not in locals() or len(visual_step_embeddings) == 0:
    print("⚠️  Visual step embeddings not loaded!")
    print("   Please run Substep 1 in extension_complete notebook first, then re-run Step 3.")
    print("   Skipping similarity computation...")
else:
    print("=" * 70)
    print("🔢 Computing Similarity Matrices")
    print("=" * 70)

    # Create projection layer to align dimensions
    # Visual: 768-dim (EgoVLP) -> 512-dim (CLIP text)
    visual_dim = 768
    text_dim = 512

    projection = nn.Linear(visual_dim, text_dim).to(DEVICE)
    projection.eval()

    print(f"📐 Created projection layer: {visual_dim} -> {text_dim}")

    # Function to compute similarity matrix for one recording
    def compute_similarity_matrix(recording_id: str, graph_name: str) -> torch.Tensor:
        """
        Compute similarity matrix between visual steps and task graph nodes.

        Args:
            recording_id: Recording ID (e.g., "1_1")
            graph_name: Task graph name (e.g., "spicytunaavocadowraps")

        Returns:
            similarity_matrix: (N_visual_steps, N_graph_nodes) cosine similarity matrix
        """
        # Get visual step embeddings
        if recording_id not in visual_step_embeddings:
            return None

        visual_emb = visual_step_embeddings[recording_id]  # (N_steps, 768)
        visual_emb = torch.from_numpy(visual_emb).float().to(DEVICE)

        # Project visual embeddings to text dimension
        with torch.no_grad():
            visual_emb_proj = projection(visual_emb)  # (N_steps, 512)
            visual_emb_proj = F.normalize(visual_emb_proj, p=2, dim=1)  # L2 normalize
            visual_emb_proj = visual_emb_proj.float()  # Ensure float32

        # Get task graph text embeddings
        if graph_name not in task_graph_embeddings:
            return None

        graph_emb_dict = task_graph_embeddings[graph_name]  # {step_id: tensor}
        graph_step_ids = sorted(graph_emb_dict.keys(), key=lambda x: int(x) if x.isdigit() else float('inf'))
        graph_emb_list = [graph_emb_dict[sid] for sid in graph_step_ids]
        graph_emb = torch.stack(graph_emb_list).to(DEVICE)  # (N_nodes, 512)
        graph_emb = graph_emb.float()  # Convert to float32 (CLIP outputs float16 by default)
        graph_emb = F.normalize(graph_emb, p=2, dim=1)  # L2 normalize

        # Compute cosine similarity
        similarity = torch.matmul(visual_emb_proj, graph_emb.t())  # (N_steps, N_nodes)

        return similarity

    # Test on a sample recording
    if len(visual_step_embeddings) > 0:
        sample_recording = list(visual_step_embeddings.keys())[0]
        # Try to find matching task graph (by recipe ID or filename)
        recipe_id = sample_recording.split('_')[0]

        # For now, use first available task graph (you'll need proper mapping)
        sample_graph = list(task_graph_embeddings.keys())[0]

        print(f"\n🧪 Testing similarity computation:")
        print(f"   Recording: {sample_recording}")
        print(f"   Task graph: {sample_graph}")

        sim_matrix = compute_similarity_matrix(sample_recording, sample_graph)
        if sim_matrix is not None:
            print(f"   ✅ Similarity matrix shape: {sim_matrix.shape}")
            print(f"   Similarity range: [{sim_matrix.min():.3f}, {sim_matrix.max():.3f}]")
            print(f"   Mean similarity: {sim_matrix.mean():.3f}")
        else:
            print(f"   ⚠️  Could not compute similarity (missing data)")

    print(f"\n✅ Similarity computation function ready!")
    print(f"   Use compute_similarity_matrix(recording_id, graph_name) for each recording")

🔢 Computing Similarity Matrices
📐 Created projection layer: 768 -> 512

🧪 Testing similarity computation:
   Recording: 1_19
   Task graph: spicytunaavocadowraps
   ✅ Similarity matrix shape: torch.Size([12, 19])
   Similarity range: [0.009, 0.141]
   Mean similarity: 0.071

✅ Similarity computation function ready!
   Use compute_similarity_matrix(recording_id, graph_name) for each recording


---

## Step 5: Hungarian Matching Algorithm

Use the Hungarian algorithm to find optimal one-to-one matching between visual steps and task graph nodes.


In [17]:
# Hungarian matching algorithm
print("=" * 70)
print("🔗 Hungarian Matching Algorithm")
print("=" * 70)

def hungarian_matching(similarity_matrix: torch.Tensor) -> Tuple[List[int], List[int], torch.Tensor]:
    """
    Find optimal one-to-one matching using Hungarian algorithm.

    Args:
        similarity_matrix: (N_visual_steps, N_graph_nodes) similarity matrix

    Returns:
        visual_indices: List of visual step indices (matched)
        graph_indices: List of graph node indices (matched)
        costs: Cost matrix (negative similarity for minimization)
    """
    # Convert to numpy for scipy
    sim_np = similarity_matrix.cpu().numpy()

    # Hungarian algorithm minimizes cost, so we use negative similarity
    cost_matrix = -sim_np

    # Apply Hungarian algorithm
    visual_indices, graph_indices = linear_sum_assignment(cost_matrix)

    # Get matched similarities
    matched_similarities = sim_np[visual_indices, graph_indices]

    return visual_indices.tolist(), graph_indices.tolist(), matched_similarities

# Test on sample similarity matrix
if 'sim_matrix' in locals() and sim_matrix is not None:
    print(f"\n🧪 Testing Hungarian matching on sample similarity matrix:")
    print(f"   Input shape: {sim_matrix.shape}")

    visual_idx, graph_idx, matched_sims = hungarian_matching(sim_matrix)

    print(f"   ✅ Found {len(visual_idx)} matches")
    print(f"   Matched visual steps: {visual_idx[:5]}{'...' if len(visual_idx) > 5 else ''}")
    print(f"   Matched graph nodes: {graph_idx[:5]}{'...' if len(graph_idx) > 5 else ''}")
    print(f"   Average matched similarity: {matched_sims.mean():.3f}")
    print(f"   Min similarity: {matched_sims.min():.3f}, Max: {matched_sims.max():.3f}")
else:
    print(f"\n⚠️  No similarity matrix available for testing")
    print(f"   Function hungarian_matching() is ready to use once embeddings are loaded")

print(f"\n✅ Hungarian matching function ready!")


🔗 Hungarian Matching Algorithm

🧪 Testing Hungarian matching on sample similarity matrix:
   Input shape: torch.Size([12, 19])
   ✅ Found 12 matches
   Matched visual steps: [0, 1, 2, 3, 4]...
   Matched graph nodes: [6, 4, 2, 9, 7]...
   Average matched similarity: 0.091
   Min similarity: 0.059, Max: 0.115

✅ Hungarian matching function ready!


---

## Step 6: Learnable Projection Layer

Create a learnable projection to combine textual and visual features for updated node representations.


In [18]:
# Learnable projection layer
print("=" * 70)
print("🧠 Learnable Projection Layer")
print("=" * 70)

class NodeFeatureProjection(nn.Module):
    """
    Learnable projection to combine textual and visual features.

    Inputs:
        - Text features: (N_nodes, 512) - CLIP text embeddings
        - Visual features: (N_matched_steps, 768) - EgoVLP visual embeddings

    Output:
        - Updated node features: (N_nodes, output_dim)
    """
    def __init__(self, text_dim=512, visual_dim=768, hidden_dim=512, output_dim=512):
        super().__init__()
        self.text_dim = text_dim
        self.visual_dim = visual_dim
        self.output_dim = output_dim

        # Project visual features to text dimension
        self.visual_proj = nn.Linear(visual_dim, text_dim)

        # Combine text and visual features
        self.combine = nn.Sequential(
            nn.Linear(text_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, text_features: torch.Tensor, visual_features: torch.Tensor,
                matched_indices: List[Tuple[int, int]]) -> torch.Tensor:
        """
        Args:
            text_features: (N_nodes, text_dim) - task graph node text embeddings
            visual_features: (N_matched_steps, visual_dim) - matched visual step embeddings
            matched_indices: List of (graph_node_idx, visual_step_idx) tuples

        Returns:
            updated_features: (N_nodes, output_dim) - updated node features
        """
        # Project visual features
        visual_proj = self.visual_proj(visual_features)  # (N_matched, text_dim)

        # Initialize output with text features
        updated = text_features.clone()  # (N_nodes, text_dim)

        # Update matched nodes
        for graph_idx, visual_idx in matched_indices:
            if graph_idx < len(updated) and visual_idx < len(visual_proj):
                # Concatenate text and visual features
                combined = torch.cat([text_features[graph_idx], visual_proj[visual_idx]], dim=0)
                # Pass through combine network
                updated[graph_idx] = self.combine(combined)

        return updated

# Create model instance
projection_model = NodeFeatureProjection(
    text_dim=512,
    visual_dim=768,
    hidden_dim=512,
    output_dim=512
).to(DEVICE)

print(f"✅ Created learnable projection model:")
print(f"   Text input: 512-dim")
print(f"   Visual input: 768-dim")
print(f"   Output: 512-dim")
print(f"   Parameters: {sum(p.numel() for p in projection_model.parameters()):,}")

# Show model structure
print(f"\n📋 Model structure:")
print(projection_model)


🧠 Learnable Projection Layer
✅ Created learnable projection model:
   Text input: 512-dim
   Visual input: 768-dim
   Output: 512-dim
   Parameters: 1,181,184

📋 Model structure:
NodeFeatureProjection(
  (visual_proj): Linear(in_features=768, out_features=512, bias=True)
  (combine): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
  )
)


---

## Step 7: Complete Pipeline - Process All Recordings

Combine all steps to process all recordings and update task graph node features.


In [19]:
# Complete pipeline function
def process_recording_with_task_graph(recording_id: str, graph_name: str,
                                      use_learnable_projection: bool = True) -> Dict:
    """
    Complete pipeline for one recording:
    1. Compute similarity matrix
    2. Apply Hungarian matching
    3. Update node features with learnable projection

    Returns:
        Dictionary with matching results and updated features
    """
    results = {
        'recording_id': recording_id,
        'graph_name': graph_name,
        'matched': False,
        'matches': [],
        'similarities': [],
        'updated_features': None
    }

    # Check if we have required data
    if recording_id not in visual_step_embeddings:
        return results

    if graph_name not in task_graph_embeddings:
        return results

    # Step 1: Compute similarity matrix
    sim_matrix = compute_similarity_matrix(recording_id, graph_name)
    if sim_matrix is None:
        return results

    # Step 2: Hungarian matching
    visual_idx, graph_idx, matched_sims = hungarian_matching(sim_matrix)

    results['matched'] = True
    results['matches'] = list(zip(visual_idx, graph_idx))
    results['similarities'] = matched_sims.tolist()

    # Step 3: Update node features (if learnable projection is enabled)
    if use_learnable_projection and len(visual_idx) > 0:
        # Get text features
        graph_emb_dict = task_graph_embeddings[graph_name]
        graph_step_ids = sorted(graph_emb_dict.keys(), key=lambda x: int(x) if x.isdigit() else float('inf'))
        text_features = torch.stack([graph_emb_dict[sid] for sid in graph_step_ids]).to(DEVICE)

        # Get matched visual features
        visual_emb = torch.from_numpy(visual_step_embeddings[recording_id]).float().to(DEVICE)
        matched_visual = visual_emb[visual_idx]

        # Create matched indices (graph_node_idx, visual_step_idx)
        matched_indices = [(graph_idx[i], i) for i in range(len(visual_idx))]

        # Update features
        with torch.no_grad():
            updated_features = projection_model(text_features, matched_visual, matched_indices)
            results['updated_features'] = updated_features.cpu().numpy()

    return results

# Test on sample recording
if 'visual_step_embeddings' in locals() and len(visual_step_embeddings) > 0:
    print("=" * 70)
    print("🧪 Testing Complete Pipeline")
    print("=" * 70)

    sample_recording = list(visual_step_embeddings.keys())[0]
    sample_graph = list(task_graph_embeddings.keys())[0]

    print(f"   Recording: {sample_recording}")
    print(f"   Task graph: {sample_graph}")

    result = process_recording_with_task_graph(sample_recording, sample_graph)

    if result['matched']:
        print(f"   ✅ Successfully matched {len(result['matches'])} steps")
        print(f"   Average similarity: {np.mean(result['similarities']):.3f}")
        if result['updated_features'] is not None:
            print(f"   Updated features shape: {result['updated_features'].shape}")
    else:
        print(f"   ⚠️  Could not process (missing data)")
else:
    print("⚠️  Visual step embeddings not loaded - cannot test pipeline")
    print("   Function process_recording_with_task_graph() is ready once embeddings are available")

print(f"\n✅ Complete pipeline function ready!")


🧪 Testing Complete Pipeline
   Recording: 1_19
   Task graph: spicytunaavocadowraps
   ✅ Successfully matched 12 steps
   Average similarity: 0.091
   Updated features shape: (19, 512)

✅ Complete pipeline function ready!


---

## Step 8: Investigate Recipe-to-Task-Graph Mapping

Before processing all recordings, let's investigate the exact mapping between recipe IDs and task graph names.

In [20]:
# Investigate recipe-to-task-graph mapping
print("=" * 70)
print("🔍 Investigating Recipe-to-Task-Graph Mapping")
print("=" * 70)

# 1. Check annotations structure
print("\n1️⃣  Checking annotations structure...")
try:
    with open(ANNOTATIONS_PATH, 'r') as f:
        annotations = json.load(f)

    # Get a sample recording
    sample_recording = list(annotations.keys())[0]
    sample_data = annotations[sample_recording]

    print(f"   Sample recording: {sample_recording}")
    print(f"   Available keys: {list(sample_data.keys())}")

    # Check for recipe-related fields
    recipe_fields = [k for k in sample_data.keys() if 'recipe' in k.lower() or 'name' in k.lower()]
    print(f"   Recipe-related fields: {recipe_fields}")

    # Show sample data structure
    if recipe_fields:
        for field in recipe_fields[:3]:
            print(f"   - {field}: {sample_data.get(field, 'N/A')}")

    # Extract all unique recipe IDs and their associated data
    recipe_id_data = {}
    for recording_id, data in annotations.items():
        recipe_id = recording_id.split('_')[0]
        if recipe_id not in recipe_id_data:
            recipe_id_data[recipe_id] = {
                'sample_recording': recording_id,
                'fields': list(data.keys()),
                'recipe_fields': {k: v for k, v in data.items() if 'recipe' in k.lower() or 'name' in k.lower()}
            }

    print(f"\n   Found {len(recipe_id_data)} unique recipe IDs")
    print(f"   Sample recipe ID data (first 3):")
    for i, (rid, data) in enumerate(list(recipe_id_data.items())[:3]):
        print(f"   - Recipe ID {rid}:")
        print(f"     Fields: {data['fields']}")
        if data['recipe_fields']:
            print(f"     Recipe fields: {data['recipe_fields']}")

except Exception as e:
    print(f"   ❌ Error loading annotations: {e}")

# 2. Check task graph files structure
print("\n2️⃣  Checking task graph files structure...")
task_graph_files = list(TASK_GRAPHS_DIR.glob("*.json")) if TASK_GRAPHS_DIR.exists() else []
print(f"   Found {len(task_graph_files)} task graph files")

if len(task_graph_files) > 0:
    # Inspect a few task graph files
    print(f"\n   Inspecting task graph files (first 3):")
    for graph_file in task_graph_files[:3]:
        try:
            with open(graph_file, 'r') as f:
                graph_data = json.load(f)

            print(f"\n   📄 {graph_file.stem}:")
            print(f"      Keys: {list(graph_data.keys())}")

            # Check for recipe-related fields
            recipe_fields = [k for k in graph_data.keys() if 'recipe' in k.lower() or 'id' in k.lower()]
            if recipe_fields:
                print(f"      Recipe-related fields: {recipe_fields}")
                for field in recipe_fields:
                    print(f"         - {field}: {graph_data.get(field, 'N/A')}")

            # Show steps structure
            if 'steps' in graph_data:
                steps = graph_data['steps']
                print(f"      Steps: {type(steps).__name__} with {len(steps)} items")
                if isinstance(steps, dict) and len(steps) > 0:
                    sample_step = list(steps.items())[0]
                    print(f"      Sample step: {sample_step[0]} -> {sample_step[1][:50] if isinstance(sample_step[1], str) else str(sample_step[1])[:50]}...")

        except Exception as e:
            print(f"      ❌ Error reading {graph_file.stem}: {e}")

# 3. Try to find mapping by checking all task graphs
print("\n3️⃣  Searching for recipe_id in all task graph files...")
recipe_id_in_graphs = {}
for graph_file in task_graph_files:
    graph_name = graph_file.stem
    try:
        with open(graph_file, 'r') as f:
            graph_data = json.load(f)

        # Check all possible fields that might contain recipe_id
        for key, value in graph_data.items():
            if 'recipe' in key.lower() and 'id' in key.lower():
                recipe_id_in_graphs[graph_name] = {
                    'field': key,
                    'value': value
                }
                print(f"   ✅ {graph_name}: Found '{key}' = {value}")
                break
    except:
        pass

if len(recipe_id_in_graphs) == 0:
    print("   ⚠️  No recipe_id fields found in task graph files")

# 4. Try to match by analyzing recording distribution
print("\n4️⃣  Analyzing recording distribution per recipe...")
if 'annotations' in locals():
    recipe_id_counts = {}
    for recording_id in annotations.keys():
        recipe_id = recording_id.split('_')[0]
        recipe_id_counts[recipe_id] = recipe_id_counts.get(recipe_id, 0) + 1

    print(f"   Recipe ID distribution (first 5):")
    for rid, count in sorted(recipe_id_counts.items(), key=lambda x: -x[1])[:5]:
        print(f"   - Recipe {rid}: {count} recordings")

# 5. Try to match by checking if task graph names appear in annotations
print("\n5️⃣  Checking if task graph names match annotation data...")
if 'annotations' in locals() and len(task_graph_files) > 0:
    task_graph_names = [f.stem.lower() for f in task_graph_files]

    # Check all annotation fields for task graph name matches
    matches_found = {}
    for recording_id, data in list(annotations.items())[:100]:  # Check first 100 for speed
        recipe_id = recording_id.split('_')[0]
        if recipe_id in matches_found:
            continue

        # Check all string fields in data
        for key, value in data.items():
            if isinstance(value, str):
                value_lower = value.lower().replace(' ', '').replace('-', '')
                for graph_name in task_graph_names:
                    graph_name_clean = graph_name.lower()
                    if graph_name_clean in value_lower or value_lower in graph_name_clean:
                        if recipe_id not in matches_found:
                            matches_found[recipe_id] = {
                                'graph_name': graph_name,
                                'matched_field': key,
                                'matched_value': value
                            }
                            print(f"   ✅ Recipe {recipe_id} -> {graph_name} (via field '{key}': '{value}')")
                            break
                if recipe_id in matches_found:
                    break

    if len(matches_found) > 0:
        print(f"\n   Found {len(matches_found)} potential matches!")
    else:
        print("   ⚠️  No matches found by string matching")

print("\n" + "=" * 70)
print("✅ Investigation complete!")
print("=" * 70)

🔍 Investigating Recipe-to-Task-Graph Mapping

1️⃣  Checking annotations structure...
   Sample recording: 1_7
   Available keys: ['recording_id', 'activity_id', 'activity_name', 'person_id', 'environment', 'steps']
   Recipe-related fields: ['activity_name']
   - activity_name: Microwave Egg Sandwich

   Found 24 unique recipe IDs
   Sample recipe ID data (first 3):
   - Recipe ID 1:
     Fields: ['recording_id', 'activity_id', 'activity_name', 'person_id', 'environment', 'steps']
     Recipe fields: {'activity_name': 'Microwave Egg Sandwich'}
   - Recipe ID 2:
     Fields: ['recording_id', 'activity_id', 'activity_name', 'person_id', 'environment', 'steps']
     Recipe fields: {'activity_name': 'Dressed Up Meatballs'}
   - Recipe ID 3:
     Fields: ['recording_id', 'activity_id', 'activity_name', 'person_id', 'environment', 'steps']
     Recipe fields: {'activity_name': 'Microwave Mug Pizza'}

2️⃣  Checking task graph files structure...
   Found 24 task graph files

   Inspecting task

---

## Step 9: Process All Recordings and Save Results

Process all recordings and save the updated task graph features.

In [21]:
# Process all recordings
if 'visual_step_embeddings' not in locals() or len(visual_step_embeddings) == 0:
    print("⚠️  Visual step embeddings not loaded!")
    print("   Please run Substep 1 first, then re-run Step 3.")
    print("   Skipping batch processing...")
else:
    print("=" * 70)
    print("🔄 Processing All Recordings")
    print("=" * 70)

    # Create proper mapping from recipe_id to task graph name
    print("\n🗺️  Creating recipe-to-task-graph mapping...")

    recipe_id_to_graph_name = {}

    # Strategy 1: Use activity_name from annotations (this is the correct field!)
    try:
        with open(ANNOTATIONS_PATH, 'r') as f:
            annotations = json.load(f)

        # Extract recipe_id -> activity_name mapping
        recipe_id_to_name = {}
        for recording_id, data in annotations.items():
            recipe_id = recording_id.split('_')[0]
            if recipe_id not in recipe_id_to_name:
                # Use activity_name field (found in investigation)
                if 'activity_name' in data:
                    activity_name = data['activity_name']
                    # Normalize: lowercase, remove spaces, hyphens, and special chars
                    normalized_name = activity_name.lower().replace(' ', '').replace('-', '').replace('_', '')
                    recipe_id_to_name[recipe_id] = normalized_name
                # Fallback to other fields if activity_name not found
                elif 'recipe_name' in data:
                    recipe_id_to_name[recipe_id] = data['recipe_name'].lower().replace(' ', '').replace('-', '')
                elif 'recipe' in data:
                    recipe_id_to_name[recipe_id] = str(data['recipe']).lower().replace(' ', '').replace('-', '')

        # Match activity names to task graph filenames
        task_graph_names = get_all_task_graph_names()

        # Create normalized mapping: normalize both activity names and graph names for matching
        def normalize_name(name):
            """Normalize name for matching: lowercase, remove spaces, hyphens, underscores, and common words"""
            normalized = name.lower().replace(' ', '').replace('-', '').replace('_', '')
            # Remove common words that might differ
            normalized = normalized.replace('the', '').replace('a', '').replace('an', '')
            return normalized

        task_graph_names_normalized = {normalize_name(name): name for name in task_graph_names}

        # First pass: exact match
        for recipe_id, activity_name in recipe_id_to_name.items():
            normalized_activity = normalize_name(activity_name)
            if normalized_activity in task_graph_names_normalized:
                recipe_id_to_graph_name[recipe_id] = task_graph_names_normalized[normalized_activity]

        # Second pass: partial/substring matching for unmapped recipes
        unmapped = set(recipe_id_to_name.keys()) - set(recipe_id_to_graph_name.keys())
        for recipe_id in unmapped:
            normalized_activity = normalize_name(recipe_id_to_name[recipe_id])
            best_match = None
            best_score = 0

            for graph_normalized, graph_name in task_graph_names_normalized.items():
                # Check if one is substring of the other
                if normalized_activity in graph_normalized:
                    score = len(normalized_activity) / len(graph_normalized)
                    if score > best_score:
                        best_score = score
                        best_match = graph_name
                elif graph_normalized in normalized_activity:
                    score = len(graph_normalized) / len(normalized_activity)
                    if score > best_score:
                        best_score = score
                        best_match = graph_name

            if best_match and best_score > 0.5:  # Only use if at least 50% match
                recipe_id_to_graph_name[recipe_id] = best_match

        print(f"   ✅ Mapped {len(recipe_id_to_graph_name)}/{len(recipe_id_to_name)} recipes from annotations (using activity_name)")
    except Exception as e:
        print(f"   ⚠️  Could not load annotations: {e}")

    # Strategy 2: Try to find recipe_id in task graph JSON files
    if len(recipe_id_to_graph_name) < len(set(r.split('_')[0] for r in visual_step_embeddings.keys())):
        print("   🔍 Checking task graph files for recipe_id...")
        task_graph_files = list(TASK_GRAPHS_DIR.glob("*.json")) if TASK_GRAPHS_DIR.exists() else []

        for graph_file in task_graph_files:
            graph_name = graph_file.stem
            try:
                with open(graph_file, 'r') as f:
                    graph_data = json.load(f)

                # Check if task graph contains recipe_id
                if 'recipe_id' in graph_data:
                    recipe_id = str(graph_data['recipe_id'])
                    if recipe_id not in recipe_id_to_graph_name:
                        recipe_id_to_graph_name[recipe_id] = graph_name
            except:
                pass

        print(f"   ✅ Found {len(recipe_id_to_graph_name)} total recipe mappings")

    # Strategy 3: Create fallback mapping by grouping recordings
    # If we still have unmapped recipes, try to infer from recording distribution
    all_recipe_ids = set(r.split('_')[0] for r in visual_step_embeddings.keys())
    unmapped_recipes = all_recipe_ids - set(recipe_id_to_graph_name.keys())

    if len(unmapped_recipes) > 0 and len(task_graph_names) > 0:
        print(f"   ⚠️  {len(unmapped_recipes)} recipes still unmapped")
        print(f"   Using fallback: assigning unmapped recipes to available task graphs")
        # Simple round-robin assignment for unmapped recipes
        unmapped_list = sorted(unmapped_recipes)
        for i, recipe_id in enumerate(unmapped_list):
            graph_idx = i % len(task_graph_names)
            recipe_id_to_graph_name[recipe_id] = task_graph_names[graph_idx]

    print(f"\n✅ Recipe-to-graph mapping complete: {len(recipe_id_to_graph_name)} recipes mapped")

    # Process all recordings with proper mapping
    all_results = {}
    processed_count = 0
    mapping_stats = defaultdict(int)

    print(f"\n📊 Processing {len(visual_step_embeddings)} recordings...")

    for recording_id in tqdm(visual_step_embeddings.keys(), desc="Processing"):
        recipe_id = recording_id.split('_')[0]

        # Get correct task graph name for this recipe
        if recipe_id in recipe_id_to_graph_name:
            graph_name = recipe_id_to_graph_name[recipe_id]
        else:
            # Fallback: use first available graph if mapping not found
            graph_name = list(task_graph_embeddings.keys())[0]
            mapping_stats['fallback'] += 1

        mapping_stats[graph_name] += 1

        result = process_recording_with_task_graph(recording_id, graph_name)
        all_results[recording_id] = result

        if result['matched']:
            processed_count += 1

    print(f"\n✅ Processed {processed_count}/{len(visual_step_embeddings)} recordings successfully")
    print(f"\n📊 Task graph usage statistics:")
    for graph_name, count in sorted(mapping_stats.items(), key=lambda x: -x[1])[:10]:
        print(f"   {graph_name}: {count} recordings")
    if mapping_stats.get('fallback', 0) > 0:
        print(f"   ⚠️  Fallback (unmapped): {mapping_stats['fallback']} recordings")

    # Save results
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    results_path = OUTPUT_DIR / "task_graph_matching_results.json"

    # Convert results to JSON-serializable format
    results_json = {}
    for rec_id, result in all_results.items():
        results_json[rec_id] = {
            'graph_name': result['graph_name'],
            'matched': result['matched'],
            'num_matches': len(result['matches']),
            'avg_similarity': float(np.mean(result['similarities'])) if result['similarities'] else None,
            'matches': result['matches'],
            'has_updated_features': result['updated_features'] is not None
        }

    with open(results_path, 'w') as f:
        json.dump(results_json, f, indent=2)

    print(f"💾 Results saved to: {results_path}")

    # Save updated features separately (if available)
    if any(r['updated_features'] is not None for r in all_results.values()):
        features_path = OUTPUT_DIR / "updated_task_graph_features.npz"
        features_dict = {
            rec_id: r['updated_features']
            for rec_id, r in all_results.items()
            if r['updated_features'] is not None
        }
        np.savez_compressed(features_path, **features_dict)
        print(f"💾 Updated features saved to: {features_path}")

🔄 Processing All Recordings

🗺️  Creating recipe-to-task-graph mapping...
   ✅ Mapped 24/24 recipes from annotations (using activity_name)

✅ Recipe-to-graph mapping complete: 24 recipes mapped

📊 Processing 383 recordings...


Processing: 100%|██████████| 383/383 [00:01<00:00, 294.33it/s]



✅ Processed 383/383 recordings successfully

📊 Task graph usage statistics:
   cucumberraita: 20 recordings
   blenderbananapancakes: 19 recordings
   microwaveeggsandwich: 18 recordings
   spicytunaavocadowraps: 18 recordings
   capresebruschetta: 18 recordings
   tomatomozzarellasalad: 17 recordings
   herbomeletwithfriedtomatoes: 17 recordings
   mugcake: 17 recordings
   ramen: 17 recordings
   scrambledeggs: 16 recordings
💾 Results saved to: extension_results/substep3/task_graph_matching_results.json
💾 Updated features saved to: extension_results/substep3/updated_task_graph_features.npz


---

## 📊 Visualization and Analysis of Results

This section creates visualizations to understand the matching results and use them in the report.

---


In [22]:
# Load Results for Visualization
print("=" * 70)
print("📊 Loading Results for Visualization")
print("=" * 70)

import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import pandas as pd

# Set style for better plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load matching results
results_path = OUTPUT_DIR / "task_graph_matching_results.json"
if results_path.exists():
    with open(results_path, 'r') as f:
        matching_results = json.load(f)
    print(f"✅ Loaded matching results for {len(matching_results)} recordings")
else:
    print(f"❌ Results file not found: {results_path}")
    matching_results = {}

# Load updated features (for statistics)
features_path = OUTPUT_DIR / "updated_task_graph_features.npz"
if features_path.exists():
    updated_features_data = np.load(features_path, allow_pickle=True)
    updated_features_dict = dict(updated_features_data)
    print(f"✅ Loaded updated features for {len(updated_features_dict)} recordings")
else:
    print(f"⚠️  Updated features file not found: {features_path}")
    updated_features_dict = {}

print(f"\n✅ Data loaded for visualization")

📊 Loading Results for Visualization
✅ Loaded matching results for 383 recordings
✅ Loaded updated features for 383 recordings

✅ Data loaded for visualization


In [24]:
# 1. Overall Statistics Visualization
print("=" * 70)
print("📈 Overall Matching Statistics")
print("=" * 70)

# Extract statistics
num_matches_list = []
avg_similarities = []
matched_count = 0
unmatched_count = 0

for rec_id, result in matching_results.items():
    if result.get('matched', False):
        matched_count += 1
        num_matches_list.append(result.get('num_matches', 0))
        if result.get('avg_similarity') is not None:
            avg_similarities.append(result['avg_similarity'])
    else:
        unmatched_count += 1

# 1. Match/Unmatch distribution - Save separately
fig1, ax1 = plt.subplots(figsize=(8, 8))
match_counts = [matched_count, unmatched_count]
labels = ['Matched', 'Unmatched']
colors = ['#2ecc71', '#e74c3c']
ax1.pie(match_counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Matching Success Rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'stat_matching_success_rate.png', dpi=300, bbox_inches='tight')
plt.close(fig1)
print(f"💾 Saved: {OUTPUT_DIR / 'stat_matching_success_rate.png'}")

# 2. Number of matches per recording - Save separately
if num_matches_list:
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    ax2.hist(num_matches_list, bins=20, color='#3498db', edgecolor='black', alpha=0.7)
    ax2.set_xlabel('Number of Matches per Recording', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.set_title('Distribution of Matches per Recording', fontsize=14, fontweight='bold')
    ax2.axvline(np.mean(num_matches_list), color='red', linestyle='--',
                label=f'Mean: {np.mean(num_matches_list):.1f}')
    ax2.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'stat_matches_per_recording.png', dpi=300, bbox_inches='tight')
    plt.close(fig2)
    print(f"💾 Saved: {OUTPUT_DIR / 'stat_matches_per_recording.png'}")

# 3. Average similarity scores - Save separately
if avg_similarities:
    fig3, ax3 = plt.subplots(figsize=(10, 6))
    ax3.hist(avg_similarities, bins=30, color='#9b59b6', edgecolor='black', alpha=0.7)
    ax3.set_xlabel('Average Similarity Score', fontsize=12)
    ax3.set_ylabel('Frequency', fontsize=12)
    ax3.set_title('Distribution of Average Similarity Scores', fontsize=14, fontweight='bold')
    ax3.axvline(np.mean(avg_similarities), color='red', linestyle='--',
                label=f'Mean: {np.mean(avg_similarities):.3f}')
    ax3.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'stat_similarity_distribution.png', dpi=300, bbox_inches='tight')
    plt.close(fig3)
    print(f"💾 Saved: {OUTPUT_DIR / 'stat_similarity_distribution.png'}")

# 4. Summary statistics table - Save separately
fig4, ax4 = plt.subplots(figsize=(8, 6))
ax4.axis('off')
stats_data = [
    ['Total Recordings', len(matching_results)],
    ['Successfully Matched', matched_count],
    ['Unmatched', unmatched_count],
    ['Match Rate', f'{100*matched_count/len(matching_results):.1f}%' if len(matching_results) > 0 else 'N/A'],
    ['Avg Matches/Recording', f'{np.mean(num_matches_list):.2f}' if num_matches_list else 'N/A'],
    ['Avg Similarity', f'{np.mean(avg_similarities):.3f}' if avg_similarities else 'N/A'],
]
table = ax4.table(cellText=stats_data, colLabels=['Metric', 'Value'],
                  cellLoc='left', loc='center', colWidths=[0.6, 0.4])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2.5)
ax4.set_title('Summary Statistics', fontweight='bold', pad=20, fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'stat_summary_table.png', dpi=300, bbox_inches='tight')
plt.close(fig4)
print(f"💾 Saved: {OUTPUT_DIR / 'stat_summary_table.png'}")

print(f"\n✅ All statistics saved separately!")

📈 Overall Matching Statistics
💾 Saved: extension_results/substep3/stat_matching_success_rate.png
💾 Saved: extension_results/substep3/stat_matches_per_recording.png
💾 Saved: extension_results/substep3/stat_similarity_distribution.png
💾 Saved: extension_results/substep3/stat_summary_table.png

✅ All statistics saved separately!


In [25]:
# 2. Task Graph Usage Distribution
print("=" * 70)
print("📊 Task Graph Usage Distribution")
print("=" * 70)

# Count how many recordings use each task graph
graph_usage = Counter()
for rec_id, result in matching_results.items():
    graph_name = result.get('graph_name', 'Unknown')
    graph_usage[graph_name] += 1

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot of top task graphs
ax1 = axes[0]
top_graphs = graph_usage.most_common(15)
graph_names = [g[0] for g in top_graphs]
graph_counts = [g[1] for g in top_graphs]

bars = ax1.barh(range(len(graph_names)), graph_counts, color='#3498db', edgecolor='black')
ax1.set_yticks(range(len(graph_names)))
ax1.set_yticklabels(graph_names, fontsize=9)
ax1.set_xlabel('Number of Recordings', fontsize=12)
ax1.set_title('Task Graph Usage (Top 15)', fontsize=14, fontweight='bold')
ax1.invert_yaxis()

# Add value labels on bars
for i, (bar, count) in enumerate(zip(bars, graph_counts)):
    ax1.text(count + 0.5, i, str(count), va='center', fontsize=9)

# Pie chart of distribution
ax2 = axes[1]
# Show top 10, group others
top_10 = graph_usage.most_common(10)
other_count = sum(count for _, count in graph_usage.items()) - sum(count for _, count in top_10)

pie_labels = [g[0] for g in top_10] + ['Others']
pie_counts = [g[1] for g in top_10] + [other_count]
colors_pie = plt.cm.Set3(range(len(pie_labels)))

ax2.pie(pie_counts, labels=pie_labels, autopct='%1.1f%%', colors=colors_pie, startangle=90)
ax2.set_title('Task Graph Distribution (Top 10 + Others)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'task_graph_usage.png', dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"💾 Saved: {OUTPUT_DIR / 'task_graph_usage.png'}")

print(f"\n📊 Total unique task graphs used: {len(graph_usage)}")
print(f"   Most used: {graph_usage.most_common(1)[0][0]} ({graph_usage.most_common(1)[0][1]} recordings)")

📊 Task Graph Usage Distribution
💾 Saved: extension_results/substep3/task_graph_usage.png

📊 Total unique task graphs used: 24
   Most used: cucumberraita (20 recordings)


In [26]:
# 3. Sample Similarity Matrix Visualization
print("=" * 70)
print("🔥 Sample Similarity Matrix Heatmaps")
print("=" * 70)

# Find a few sample recordings with good matches
sample_recordings = []
for rec_id, result in matching_results.items():
    if result.get('matched', False) and result.get('num_matches', 0) > 0:
        sample_recordings.append((rec_id, result))
        if len(sample_recordings) >= 3:
            break

if len(sample_recordings) == 0:
    print("⚠️  No matched recordings found for visualization")
else:
    fig, axes = plt.subplots(1, len(sample_recordings), figsize=(6*len(sample_recordings), 5))
    if len(sample_recordings) == 1:
        axes = [axes]

    for idx, (rec_id, result) in enumerate(sample_recordings):
        # Recompute similarity matrix for visualization
        graph_name = result.get('graph_name')
        if graph_name and rec_id in visual_step_embeddings:
            sim_matrix = compute_similarity_matrix(rec_id, graph_name)
            if sim_matrix is not None:
                sim_matrix_np = sim_matrix.cpu().numpy()

                # Create heatmap
                im = axes[idx].imshow(sim_matrix_np, cmap='YlOrRd', aspect='auto',
                                     vmin=0, vmax=1, interpolation='nearest')
                axes[idx].set_xlabel('Task Graph Nodes', fontsize=11)
                axes[idx].set_ylabel('Visual Steps', fontsize=11)
                axes[idx].set_title(f'Recording: {rec_id}\nGraph: {graph_name[:20]}...',
                                   fontsize=10, fontweight='bold')

                # Add colorbar
                plt.colorbar(im, ax=axes[idx], label='Similarity Score')

                # Highlight matched pairs
                matches = result.get('matches', [])
                for step_idx, node_idx in matches:
                    axes[idx].scatter(node_idx, step_idx, color='cyan', marker='x',
                                    s=100, linewidths=2, label='Matched' if step_idx == matches[0][0] else '')

                # Add grid
                axes[idx].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'sample_similarity_matrices.png', dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"💾 Saved: {OUTPUT_DIR / 'sample_similarity_matrices.png'}")

🔥 Sample Similarity Matrix Heatmaps
💾 Saved: extension_results/substep3/sample_similarity_matrices.png


In [27]:
# 4. Similarity Score Distribution Analysis
print("=" * 70)
print("📊 Similarity Score Analysis")
print("=" * 70)

# Collect all similarity scores from matches
all_similarities = []
for rec_id, result in matching_results.items():
    if result.get('matched', False):
        matches = result.get('matches', [])
        # Get individual similarity scores if available
        # (We'll need to recompute or store them)
        if result.get('avg_similarity') is not None:
            all_similarities.append(result['avg_similarity'])

if all_similarities:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram with KDE
    ax1 = axes[0]
    ax1.hist(all_similarities, bins=40, color='#3498db', edgecolor='black',
             alpha=0.7, density=True, label='Distribution')

    # Add KDE curve
    from scipy import stats
    if len(all_similarities) > 1:
        kde = stats.gaussian_kde(all_similarities)
        x_range = np.linspace(min(all_similarities), max(all_similarities), 200)
        ax1.plot(x_range, kde(x_range), 'r-', linewidth=2, label='KDE')

    ax1.axvline(np.mean(all_similarities), color='red', linestyle='--',
                linewidth=2, label=f'Mean: {np.mean(all_similarities):.3f}')
    ax1.axvline(np.median(all_similarities), color='green', linestyle='--',
                linewidth=2, label=f'Median: {np.median(all_similarities):.3f}')
    ax1.set_xlabel('Average Similarity Score', fontsize=12)
    ax1.set_ylabel('Density', fontsize=12)
    ax1.set_title('Similarity Score Distribution', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Box plot
    ax2 = axes[1]
    bp = ax2.boxplot([all_similarities], vert=True, patch_artist=True,
                     labels=['Similarity Scores'], showmeans=True)
    bp['boxes'][0].set_facecolor('#3498db')
    bp['boxes'][0].set_alpha(0.7)
    ax2.set_ylabel('Similarity Score', fontsize=12)
    ax2.set_title('Similarity Score Box Plot', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')

    # Add statistics text
    stats_text = f'Mean: {np.mean(all_similarities):.3f}\n'
    stats_text += f'Median: {np.median(all_similarities):.3f}\n'
    stats_text += f'Std: {np.std(all_similarities):.3f}\n'
    stats_text += f'Min: {np.min(all_similarities):.3f}\n'
    stats_text += f'Max: {np.max(all_similarities):.3f}'
    ax2.text(1.15, 0.5, stats_text, transform=ax2.transAxes,
            fontsize=10, verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'similarity_distribution.png', dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"💾 Saved: {OUTPUT_DIR / 'similarity_distribution.png'}")

    print(f"\n📊 Similarity Statistics:")
    print(f"   Mean: {np.mean(all_similarities):.3f}")
    print(f"   Median: {np.median(all_similarities):.3f}")
    print(f"   Std: {np.std(all_similarities):.3f}")
    print(f"   Range: [{np.min(all_similarities):.3f}, {np.max(all_similarities):.3f}]")
else:
    print("⚠️  No similarity scores available for analysis")

📊 Similarity Score Analysis


/tmp/ipython-input-2763822169.py:43: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax2.boxplot([all_similarities], vert=True, patch_artist=True,


💾 Saved: extension_results/substep3/similarity_distribution.png

📊 Similarity Statistics:
   Mean: 0.078
   Median: 0.080
   Std: 0.014
   Range: [0.026, 0.108]


In [28]:
# 5. Create Summary Report Table
print("=" * 70)
print("📋 Generating Summary Report")
print("=" * 70)

# Create comprehensive summary
summary_data = []

for rec_id, result in list(matching_results.items())[:20]:  # First 20 for display
    summary_data.append({
        'Recording ID': rec_id,
        'Graph Name': result.get('graph_name', 'N/A')[:30],
        'Matched': 'Yes' if result.get('matched', False) else 'No',
        'Num Matches': result.get('num_matches', 0),
        'Avg Similarity': f"{result.get('avg_similarity', 0):.3f}" if result.get('avg_similarity') else 'N/A',
        'Has Updated Features': 'Yes' if result.get('has_updated_features', False) else 'No'
    })

# Create DataFrame
df_summary = pd.DataFrame(summary_data)

# Save to CSV
csv_path = OUTPUT_DIR / 'matching_summary_table.csv'
df_summary.to_csv(csv_path, index=False)
print(f"💾 Saved summary table: {csv_path}")

# Display table
print(f"\n📊 Sample Summary (first 20 recordings):")
print(df_summary.to_string(index=False))

# Create overall statistics
overall_stats = {
    'Total Recordings': len(matching_results),
    'Matched Recordings': sum(1 for r in matching_results.values() if r.get('matched', False)),
    'Unmatched Recordings': sum(1 for r in matching_results.values() if not r.get('matched', False)),
    'Total Matches': sum(r.get('num_matches', 0) for r in matching_results.values()),
    'Avg Matches per Recording': np.mean([r.get('num_matches', 0) for r in matching_results.values()]),
    'Avg Similarity (matched)': np.mean([r.get('avg_similarity', 0) for r in matching_results.values()
                                         if r.get('avg_similarity') is not None]),
    'Recordings with Updated Features': sum(1 for r in matching_results.values()
                                            if r.get('has_updated_features', False)),
}

# Save overall stats
stats_path = OUTPUT_DIR / 'overall_statistics.json'
with open(stats_path, 'w') as f:
    json.dump(overall_stats, f, indent=2)

print(f"\n📊 Overall Statistics:")
for key, value in overall_stats.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.3f}")
    else:
        print(f"   {key}: {value}")

print(f"\n💾 Saved overall statistics: {stats_path}")

📋 Generating Summary Report
💾 Saved summary table: extension_results/substep3/matching_summary_table.csv

📊 Sample Summary (first 20 recordings):
Recording ID           Graph Name Matched  Num Matches Avg Similarity Has Updated Features
        1_19 microwaveeggsandwich     Yes           12          0.081                  Yes
        1_14 microwaveeggsandwich     Yes           12          0.086                  Yes
         1_7 microwaveeggsandwich     Yes           12          0.062                  Yes
        1_32 microwaveeggsandwich     Yes           12          0.063                  Yes
        1_33 microwaveeggsandwich     Yes           12          0.078                  Yes
        1_34 microwaveeggsandwich     Yes           12          0.049                  Yes
        1_36 microwaveeggsandwich     Yes           12          0.070                  Yes
        1_10 microwaveeggsandwich     Yes           12          0.086                  Yes
        1_28 microwaveeggsandwich  